# 07: Retraining and promotion (ADR-009)

Flow: **drift verdict -> retrain -> compare with the champion -> promote only if clearly better -> export for deploy.**

1. Read the drift verdict saved by notebook 06 (`reports/drift/drift_shifted.json`). Retrain only on `drift`.
2. Build a candidate: an unfitted clone of the champion (same preprocessing and hyperparameters), refit on the original training data **plus newly labelled rows**.
3. Score champion and candidate on two sets neither has seen: **shifted** data (the new reality) and **natural** data (guard against forgetting the old population).
4. Promote only if ROC AUC on shifted data improves by >= 0.005 **and** ROC AUC on natural data drops by no more than 0.01. (ROC AUC, not F1: F1 moves with the churn rate, as Phase 7 showed.)
5. If promoted: register a new model version, move the `champion` alias, export `champion.zip` for the repo. If not: the run is logged but nothing is registered.

**How "new labelled data" is simulated.** This project has no live customers, so the held-out *test* set plays the role of newly arrived, labelled data. It is split 50/50 (stratified) *before* any resampling: one half supplies new training rows, the other half is evaluation-only. Each half is then resampled toward the shifted population separately, so no row appears on both sides. The consequence, stated plainly: **the original test set is consumed by this notebook** and is no longer an untouched final test. A real deployment would use genuinely new labelled data.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml
EVIDENTLY_SPEC = "evidently>=0.7"   # HTML report only; the verdict comes from ai_experiment_lab.monitoring

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
import json, shutil
from pathlib import Path
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split

from ai_experiment_lab.data import load_csv
from ai_experiment_lab.preprocessing import split_data
from ai_experiment_lab.registry import export_pipeline, load_exported
from ai_experiment_lab.registry.mlflow_registry import alias_version, load_by_alias, register_model, run_summary
from ai_experiment_lab.retraining import decide_promotion, retrain_candidate, score, shifted_sample, should_retrain
from ai_experiment_lab.tracking import backup_tracking_db, get_code_version, library_versions, setup_tracking
from ai_experiment_lab.tracking.mlflow_logger import trusted_types
from ai_experiment_lab.utils.config import get_data_dir, load_config

cfg = load_config(f"{REPO_DIR}/configs/default.yaml")
d, sp, seed = cfg["dataset"], cfg["split"], cfg["project"]["random_seed"]
EXPERIMENT = cfg["tracking"]["experiment_name"]
REGISTERED_NAME = "telco-churn-classifier"
SHIFT_STRENGTH = 10            # the strength notebook 06 flagged as drift
FORCE = False                  # True = retrain even without a drift verdict

setup_tracking(EXPERIMENT)
df, info = load_csv(Path(get_data_dir()) / d["file"])
split = split_data(df, d["target"], d["positive_label"], d["id_columns"], sp["test_size"], seed, sp["stratify"])
champion = load_by_alias(REGISTERED_NAME)
champ_mv = alias_version(REGISTERED_NAME)
print("current champion: registry version", champ_mv.version, "| source run", champ_mv.tags.get("source_run_id"))

drift_file = Path(DRIVE_ROOT) / "reports" / "drift" / "drift_shifted.json"
verdict = json.loads(drift_file.read_text())["verdict"]
go, why = should_retrain(verdict, force=FORCE)
print(f"drift verdict: {verdict} -> retrain: {go} ({why})")
assert go, "No retraining needed. Set FORCE = True to retrain anyway."

## 1. Simulate newly arrived labelled data (split first, then shift each half)

In [ ]:
pool_X, pool_y = split.X_test, split.y_test
fit_X, eval_X, fit_y, eval_y = train_test_split(pool_X, pool_y, test_size=0.5, stratify=pool_y, random_state=seed)

new_X = shifted_sample(fit_X, SHIFT_STRENGTH, n=len(fit_X), seed=1)          # new training rows (shifted)
new_y = fit_y.loc[new_X.index]
eval_shift_X = shifted_sample(eval_X, SHIFT_STRENGTH, n=len(eval_X), seed=2)  # evaluation, shifted
eval_shift_y = eval_y.loc[eval_shift_X.index]
eval_nat_X, eval_nat_y = eval_X, eval_y                                       # evaluation, natural

assert not set(fit_X.index) & set(eval_X.index)      # no row on both sides
print(f"original training rows: {len(split.X_train):,} | new training rows: {len(new_X):,}")
print(f"eval shifted: {len(eval_shift_X):,} (churn {eval_shift_y.mean():.3f}) | eval natural: {len(eval_nat_X):,} (churn {eval_nat_y.mean():.3f})")

## 2. Build the candidate and compare

In [ ]:
candidate = retrain_candidate(champion, split.X_train, split.y_train, new_X, new_y)

rows = {}
for label, model in [("champion", champion), ("candidate", candidate)]:
    rows[(label, "shifted")] = score(model, eval_shift_X, eval_shift_y)
    rows[(label, "natural")] = score(model, eval_nat_X, eval_nat_y)
table = pd.DataFrame(rows).T.round(4)
table.index.names = ["model", "eval data"]
table

In [ ]:
decision = decide_promotion(rows[("champion", "shifted")], rows[("candidate", "shifted")],
                            rows[("champion", "natural")], rows[("candidate", "natural")])
print(decision.summary())

## 3. Log the candidate run (always) and register it (only if promoted)
The run is logged either way, so a rejected retrain leaves a record of what was tried and why it lost.

In [ ]:
with mlflow.start_run(run_name="retrain-" + REGISTERED_NAME) as run:
    mlflow.log_params({
        "retrain.trigger": verdict, "retrain.forced": FORCE,
        "retrain.champion_version": int(champ_mv.version),
        "retrain.new_rows": len(new_X), "retrain.shift_strength": SHIFT_STRENGTH,
        "model.name": "logistic_regression", "seed": seed,
    })
    mlflow.set_tags({
        "stage": "retrained", "dataset.name": info.name, "dataset.sha256": info.sha256,
        "code.version": get_code_version(), "retrain.promoted": str(decision.promote),
        "adr": "ADR-009",
        **{f"lib.{k}": v for k, v in library_versions().items()},
    })
    for (label, where), m in rows.items():
        mlflow.log_metrics({f"{label}_{where}_{k}": v for k, v in m.items()})
    mlflow.log_metrics({f"test_{k}": v for k, v in rows[("candidate", "shifted")].items()})   # shape register/export expect
    mlflow.log_dict(decision.__dict__, "promotion_decision.json")
    sample = split.X_train.head(5)
    mlflow.sklearn.log_model(candidate, name="model", signature=infer_signature(sample, candidate.predict(sample)),
                             input_example=sample, skops_trusted_types=trusted_types(candidate))
    run_id = run.info.run_id
print("candidate run:", run_id, "| promoted:", decision.promote)

if decision.promote:
    version = register_model(run_id, REGISTERED_NAME, alias="champion",
                             tags={"adr": "ADR-009", "retrain.trigger": verdict,
                                   "retrain.previous_champion": str(champ_mv.version)})
    mv = alias_version(REGISTERED_NAME)
    print(f"registered version {version}; alias 'champion' -> version {mv.version} (was {champ_mv.version})")
else:
    print("not registered; champion stays at version", champ_mv.version)

## 4. Export the new champion for the repo (only if promoted)

In [ ]:
if decision.promote:
    out_dir = Path("/content/champion_export")
    shutil.rmtree(out_dir, ignore_errors=True)
    reloaded = load_by_alias(REGISTERED_NAME)                     # exactly what the registry now serves
    assert np.allclose(reloaded.predict_proba(eval_nat_X), candidate.predict_proba(eval_nat_X))
    summary = run_summary(run_id)
    meta = {**summary, "registered_name": REGISTERED_NAME, "registry_version": int(mv.version),
            "alias": "champion", "positive_label": d["positive_label"], "threshold": 0.5,
            "adr": "ADR-009", "retrain_trigger": verdict, "previous_version": int(champ_mv.version),
            "promotion": {"gain_shifted": decision.gain_shifted, "change_natural": decision.change_natural}}
    export_pipeline(reloaded, out_dir, meta)
    pipe2, meta2 = load_exported(out_dir)
    assert np.allclose(pipe2.predict_proba(eval_nat_X), candidate.predict_proba(eval_nat_X))
    print("exported:", sorted(p.name for p in out_dir.iterdir()), "| registry_version:", meta2["registry_version"])
    drive_copy = Path(DRIVE_ROOT) / "export" / f"champion_v{mv.version}"
    shutil.copytree(out_dir, drive_copy, dirs_exist_ok=True)
    zip_path = shutil.make_archive("/content/champion", "zip", out_dir)
    print("copy on Drive:", drive_copy, "| zip:", zip_path)
else:
    print("nothing to export")

In [ ]:
report = {"verdict": verdict, "promoted": decision.promote, "reasons": decision.reasons,
          "champion_version_before": int(champ_mv.version), "candidate_run": run_id,
          "metrics": {f"{a}/{b}": m for (a, b), m in rows.items()}}
(Path(DRIVE_ROOT) / "reports").mkdir(exist_ok=True, parents=True)
(Path(DRIVE_ROOT) / "reports" / "retrain_report.json").write_text(json.dumps(report, indent=2, default=str))
print("backup:", backup_tracking_db())
if decision.promote:
    from google.colab import files
    files.download(zip_path)       # becomes models/champion/ in the repo

## What to paste back
The output of cells 3 (champion + verdict), 4 (split sizes), 5 (comparison table), 6 (decision), 7 (run and registry lines) and 8 (exported files). Then: unzip `champion.zip` over `models/champion/`, commit, and run the **Redeploy** workflow on GitHub (docs/retraining.md).